# Objective

A gradio chatbot expert in basic math

In [ ]:
import os
from labs.llm_utils import load_api_keys, get_llm_client
import gradio as gr

load_api_keys()

llm_client = get_llm_client()



Utility class to represent the chatbot 

In [11]:
from dataclasses import dataclass


class Chatbot:

    def __init__(self, name, llm_client, model, system_prompt) -> None:
        self.name = name
        self.llm_client = llm_client
        self.model = model
        self.system_prompt = system_prompt

    def chat(self, message, history):
        history = [{"role":h["role"], "content":h["content"]} for h in history]
        messages = [{"role": "system", "content": self.system_prompt}] + history + [{"role": "user", "content": message}]
        stream = self.llm_client.chat.completions.create(model=self.model, messages=messages, stream=True)
        response = ""
        for chunk in stream:
            response += chunk.choices[0].delta.content or ''
            yield response


chatbot_guidelines = """
Expert in answer basic math questions. 
Do not reply to anything that is not related to a basic math question. 
You can only reply in English or Spanish based on last message language.
"""


chatbot = Chatbot(name="Wally", llm_client=llm_client, model="gpt-4.1-mini", system_prompt=chatbot_guidelines)

Launch the chatbot UI using gradio

In [ ]:
def check_credentials(username, password):
    current_username = os.getenv("CHATBOT_USER")
    current_password = os.getenv("CHATBOT_PASS")
    return current_password is not None and current_username is not None and username == current_username and password == current_password

demo = gr.ChatInterface(
    fn=chatbot.chat,
    title=chatbot.name,
    description=f"Basic math assitant using {chatbot.model}",
    type="messages")

demo.launch(
    auth=check_credentials,
    width="75%",
    height=300,
    share=True
)
    